# Response Analysis: Transient and Steady-State

**Learning Goals**
- Distinguish between transient and steady-state response
- Understand how DC gain and time constant affect first-order system response
- Understand how DC gain, natural frequency, and damping ratio affect second-order system response
- Analyze mass-spring-damper response characteristics

## Relevant lecture video

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/b4310420-c828-4c20-85a5-050df7c53d45/public?autoplay=false&automute=false&currentMediaId=cf4ec018-c7b1-4cd3-9b01-c4801bb0668a" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [1]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded.")

Libraries loaded.


---
## Transient vs Steady-State Response

The complete response of a system can be split into two parts:

| Component | Description |
|-----------|-------------|
| **Transient response** | The part that decays to zero as $t \to \infty$. Governed by the system's poles. |
| **Steady-state response** | The part that remains as $t \to \infty$. Follows the input shape (scaled by DC gain). |

$$y(t) = y_{\text{transient}}(t) + y_{\text{ss}}(t)$$

For a **stable** system, all transient terms die out exponentially. The speed of decay depends on how far left the poles are in the $s$-plane.

---
## First-Order System Response

Recall the standard first-order system:

$$H(s) = \frac{K}{\tau s + 1}$$

where $K$ is the DC gain and $\tau$ is the time constant.

**Step response**: $y(t) = K\bigl(1 - e^{-t/\tau}\bigr)$

- Transient: $-Ke^{-t/\tau}$
- Steady state: $K$

Use the sliders below to see how $K$ and $\tau$ independently affect the response.

In [3]:
K_1d = widgets.FloatSlider(min=0.2, max=5, step=0.1, value=1,
                           description='K (DC gain):',
                           style={'description_width': 'initial'})
tau_1d = widgets.FloatSlider(min=0.2, max=5, step=0.1, value=1,
                             description=r'$\\tau$ (s):',
                             style={'description_width': 'initial'})
run_1d = widgets.Button(description='Plot', button_style='primary')
out_1d = widgets.Output()

def plot_1st_order(K, tau):
    t = np.linspace(0, 5 * tau, 500)
    y = K * (1 - np.exp(-t / tau))

    fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
    ax.plot(t, np.ones_like(t), 'k--', linewidth=1, alpha=0.3, label='Input (step)')
    ax.plot(t, y, 'b-', linewidth=2, label='Response')
    ax.axhline(K, color='gray', linestyle=':', alpha=0.5, label=f'Steady state = {K}')
    ax.axvline(tau, color='r', linestyle=':', alpha=0.4, label=f'$\\tau$ = {tau} s')
    ax.set_title(f'First-Order Step Response  (K={K}, $\\tau$={tau})')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('y(t)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

def on_run_1d(b):
    with out_1d:
        clear_output(wait=True)
        plot_1st_order(K_1d.value, tau_1d.value)

run_1d.on_click(on_run_1d)
display(widgets.VBox([K_1d, tau_1d, run_1d, out_1d]))
print("Adjust K and tau, then click Plot.")

Adjust K and tau, then click Plot.


### How DC gain changes the response

The DC gain $K$ scales the entire response vertically:
- **Transient**: amplitude scales by $K$ (but the shape and time constant are unchanged)
- **Steady state**: final value changes proportionally to $K$

Set $\tau = 1$ and vary $K$ to see that the response reaches $K$ after $5\tau$, and the $63.2\%$ mark is always at $K \times 0.632$.

### How the time constant changes the response

The time constant $\tau$ controls the speed:
- **Transient**: larger $\tau$ slows down the exponential decay, therefore the system takes longer to settle
- **Steady state**: unaffected by $\tau$ (the final value is $K$ regardless)

Set $K = 1$ and vary $\tau$ to see the response stretch or compress horizontally.

In [4]:
K_1d_comp = widgets.FloatSlider(min=0.2, max=5, step=0.1, value=1,
                                description='K (DC gain):',
                                style={'description_width': 'initial'})
tau_1d_comp = widgets.FloatSlider(min=0.2, max=5, step=0.1, value=1,
                                  description=r'$\\tau$ (s):',
                                  style={'description_width': 'initial'})
param_1d = widgets.Dropdown(options=['Vary K (fixed tau)', 'Vary tau (fixed K)'],
                            value='Vary K (fixed tau)', description='Sweep:',
                            style={'description_width': 'initial'})
run_1d_comp = widgets.Button(description='Plot', button_style='primary')
out_1d_comp = widgets.Output()

def plot_1st_order_compare(K, tau, mode):
    t = np.linspace(0, 8, 500)
    fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))

    if mode == 'Vary K (fixed tau)':
        tau_fixed = tau
        K_vals = [0.5, 1, 2, 4]
        for Kv in K_vals:
            y = Kv * (1 - np.exp(-t / tau_fixed))
            ax.plot(t, y, linewidth=1.5, label=f'K = {Kv}')
        ax.set_title(f'Varying K with $\\tau$ = {tau_fixed} s')
    else:
        K_fixed = K
        tau_vals = [0.5, 1, 2, 4]
        for tv in tau_vals:
            y = K_fixed * (1 - np.exp(-t / tv))
            ax.plot(t, y, linewidth=1.5, label=f'$\\tau$ = {tv} s')
        ax.set_title(f'Varying $\\tau$ with K = {K_fixed}')

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('y(t)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

def on_run_1d_comp(b):
    with out_1d_comp:
        clear_output(wait=True)
        plot_1st_order_compare(K_1d_comp.value, tau_1d_comp.value, param_1d.value)

run_1d_comp.on_click(on_run_1d_comp)
display(widgets.VBox([K_1d_comp, tau_1d_comp, param_1d, run_1d_comp, out_1d_comp]))
print("Select sweep mode, adjust reference, then click Plot.")

Select sweep mode, adjust reference, then click Plot.


---
## Second-Order System Response

Standard form:

$$H(s) = \frac{K \omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

For the underdamped case ($0 < \zeta < 1$):

**Step response**:
$$y(t) = K\left[1 - \frac{e^{-\zeta\omega_n t}}{\sqrt{1-\zeta^2}} \sin(\omega_d t + \phi)\right], \quad
\phi = \cos^{-1}(\zeta), \quad \omega_d = \omega_n\sqrt{1-\zeta^2}$$

Use the interactive plot below to explore how each parameter changes the response.

In [5]:
K_2d = widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1,
                           description='K (DC gain):',
                           style={'description_width': 'initial'})
wn_2d = widgets.FloatSlider(min=0.5, max=10, step=0.1, value=2,
                            description=r'$\\omega_n$ (rad/s):',
                            style={'description_width': 'initial'})
zeta_2d = widgets.FloatSlider(min=0.05, max=2, step=0.05, value=0.3,
                              description=r'$\\zeta$:',
                              style={'description_width': 'initial'})
run_2d_int = widgets.Button(description='Plot', button_style='primary')
out_2d_int = widgets.Output()

def plot_2nd_order_interactive(K, wn, zeta):
    wd = wn * np.sqrt(abs(1 - zeta**2)) if zeta != 1 else 0
    t_max = max(10, 6 / (zeta * wn)) if zeta > 0 else 20
    t = np.linspace(0, t_max, 1000)

    if zeta == 0:
        y = K * (1 - np.cos(wn * t))
    elif zeta == 1:
        y = K * (1 - (1 + wn * t) * np.exp(-wn * t))
    elif zeta > 1:
        sd = wn * np.sqrt(zeta**2 - 1)
        p1 = -zeta * wn + sd
        p2 = -zeta * wn - sd
        y = K * (1 - (p2 * np.exp(p1 * t) - p1 * np.exp(p2 * t)) / (p2 - p1))
    else:
        phi = np.arccos(zeta)
        y = K * (1 - np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2) * np.sin(wd * t + phi))

    # Envelope curves
    env_upper = K * (1 + np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2)) if zeta < 1 else None
    env_lower = K * (1 - np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2)) if zeta < 1 else None

    fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
    ax.plot(t, np.ones_like(t), 'k--', linewidth=1, alpha=0.3, label='Input (step)')
    ax.plot(t, y, 'b-', linewidth=2, label='Response')
    ax.axhline(K, color='gray', linestyle=':', alpha=0.5, label=f'Steady state = {K}')
    if env_upper is not None:
        ax.plot(t, env_upper, 'r--', linewidth=0.8, alpha=0.4, label='Envelope')
        ax.plot(t, env_lower, 'r--', linewidth=0.8, alpha=0.4)

    info = f'$\\omega_d$ = {wd:.3f} rad/s,  $\\sigma$ = {zeta*wn:.3f}'
    ax.text(0.98, 0.05, info, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title(f'Second-Order Step Response (K={K}, $\\omega_n$={wn}, $\\zeta$={zeta})')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('y(t)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def on_run_2d_int(b):
    with out_2d_int:
        clear_output(wait=True)
        plot_2nd_order_interactive(K_2d.value, wn_2d.value, zeta_2d.value)

run_2d_int.on_click(on_run_2d_int)
display(widgets.VBox([K_2d, wn_2d, zeta_2d, run_2d_int, out_2d_int]))
print("Adjust K, omega_n, and zeta, then click Plot.")

Adjust K, omega_n, and zeta, then click Plot.


### How each parameter affects the response

| Change | Effect on transient | Effect on steady state |
|--------|-------------------|----------------------|
| **Increase $K$** | Oscillation amplitude scales up | Final value increases proportionally |
| **Increase $\omega_n$** | Faster oscillations, shorter settling time | Unchanged (still $K$) |
| **Increase $\zeta$** (within $0<\zeta<1$) | Oscillations decay faster, overshoot decreases | Unchanged |
| $\zeta \to 0$ | Sustained oscillation at $\omega_n$ | Never reaches steady state |
| $\zeta \ge 1$ | No oscillation; slower as $\zeta$ grows | Unchanged |

Try these experiments:
1. **Fix $\omega_n=2$, $\zeta=0.3$** and sweep $K$ from $0.5$ to $4$
2. **Fix $K=1$, $\zeta=0.3$** and sweep $\omega_n$ from $1$ to $8$
3. **Fix $K=1$, $\omega_n=2$** and sweep $\zeta$ from $0.1$ to $1.5$

In [6]:
K_2d_sweep = widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1,
                                 description='K (DC gain):',
                                 style={'description_width': 'initial'})
wn_2d_sweep = widgets.FloatSlider(min=0.5, max=10, step=0.1, value=2,
                                  description=r'$\\omega_n$ (rad/s):',
                                  style={'description_width': 'initial'})
zeta_2d_sweep = widgets.FloatSlider(min=0.05, max=2, step=0.05, value=0.3,
                                    description=r'$\\zeta$:',
                                    style={'description_width': 'initial'})
param_2d = widgets.Dropdown(
    options=['Vary K (fixed wn, zeta)', 'Vary wn (fixed K, zeta)', 'Vary zeta (fixed K, wn)'],
    value='Vary K (fixed wn, zeta)', description='Sweep:',
    style={'description_width': 'initial'})
run_2d_sweep = widgets.Button(description='Plot', button_style='primary')
out_2d_sweep = widgets.Output()

def plot_2nd_order_sweep(K, wn, zeta, mode):
    t_max = 12
    t = np.linspace(0, t_max, 1000)
    fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))

    def step_2nd(Kv, wnv, zetav, t):
        wd = wnv * np.sqrt(abs(1 - zetav**2)) if zetav != 1 else 0
        if zetav == 0:
            return Kv * (1 - np.cos(wnv * t))
        elif zetav == 1:
            return Kv * (1 - (1 + wnv * t) * np.exp(-wnv * t))
        elif zetav > 1:
            sd = wnv * np.sqrt(zetav**2 - 1)
            p1 = -zetav * wnv + sd
            p2 = -zetav * wnv - sd
            return Kv * (1 - (p2 * np.exp(p1 * t) - p1 * np.exp(p2 * t)) / (p2 - p1))
        else:
            phi = np.arccos(zetav)
            return Kv * (1 - np.exp(-zetav * wnv * t) / np.sqrt(1 - zetav**2) * np.sin(wd * t + phi))

    if mode == 'Vary K (fixed wn, zeta)':
        K_vals = [0.5, 1, 2, 4]
        for Kv in K_vals:
            y = step_2nd(Kv, wn, zeta, t)
            ax.plot(t, y, linewidth=1.5, label=f'K = {Kv}')
        ax.set_title(f'Varying K  ($\\omega_n$={wn}, $\\zeta$={zeta})')
        ax.axhline(K, color='gray', linestyle=':', alpha=0.3)
    elif mode == 'Vary wn (fixed K, zeta)':
        wn_vals = [1, 2, 4, 8]
        for wnv in wn_vals:
            y = step_2nd(K, wnv, zeta, t)
            ax.plot(t, y, linewidth=1.5, label=f'$\\omega_n$ = {wnv}')
        ax.set_title(f'Varying $\\omega_n$  (K={K}, $\\zeta$={zeta})')
    else:
        zeta_vals = [0.15, 0.4, 0.7, 1.0, 1.5]
        for zetav in zeta_vals:
            y = step_2nd(K, wn, zetav, t)
            label = f'$\\zeta$ = {zetav}'
            ax.plot(t, y, linewidth=1.5, label=label)
        ax.set_title(f'Varying $\\zeta$  (K={K}, $\\omega_n$={wn})')

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('y(t)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

def on_run_2d_sweep(b):
    with out_2d_sweep:
        clear_output(wait=True)
        plot_2nd_order_sweep(K_2d_sweep.value, wn_2d_sweep.value, zeta_2d_sweep.value, param_2d.value)

run_2d_sweep.on_click(on_run_2d_sweep)
display(widgets.VBox([K_2d_sweep, wn_2d_sweep, zeta_2d_sweep, param_2d, run_2d_sweep, out_2d_sweep]))
print("Select sweep mode, adjust reference parameters, then click Plot.")

Select sweep mode, adjust reference parameters, then click Plot.


---
## Mass-Spring-Damper: Parameter Exploration

The mass-spring-damper maps to standard 2nd-order form via:

$$\omega_n = \sqrt{\frac{k}{m}}, \qquad
\zeta = \frac{b}{2\sqrt{km}}, \qquad
K = \frac{1}{k}$$

This means changing $m$, $b$, or $k$ affects multiple standard-form parameters simultaneously:

| Change | Effect on $K$ | Effect on $\omega_n$ | Effect on $\zeta$ |
|--------|--------------|---------------------|------------------|
| Increase $m$ | None | Decreases | Decreases |
| Increase $b$ | None | None | Increases |
| Increase $k$ | Decreases | Increases | Decreases |

Explore the combined effect with the interactive plot below.

In [7]:
m_msd2 = widgets.FloatSlider(min=0.5, max=10, step=0.1, value=2,
                             description='m (kg):',
                             style={'description_width': 'initial'})
b_msd2 = widgets.FloatSlider(min=0.1, max=10, step=0.1, value=1,
                             description='b (N\u00b7s/m):',
                             style={'description_width': 'initial'})
k_msd2 = widgets.FloatSlider(min=0.5, max=20, step=0.1, value=5,
                             description='k (N/m):',
                             style={'description_width': 'initial'})
run_msd2 = widgets.Button(description='Plot', button_style='primary')
out_msd2 = widgets.Output()

def plot_msd_response(m, b, k):
    wn = np.sqrt(k / m)
    zeta = b / (2 * np.sqrt(k * m))
    K = 1 / k
    wd = wn * np.sqrt(abs(1 - zeta**2)) if zeta != 1 else 0
    sigma = zeta * wn

    t_max = max(10, 6 / sigma) if sigma > 0 else 30
    t = np.linspace(0, t_max, 1000)

    if zeta == 0:
        y = K * (1 - np.cos(wn * t))
    elif zeta == 1:
        y = K * (1 - (1 + wn * t) * np.exp(-wn * t))
    elif zeta > 1:
        sd = wn * np.sqrt(zeta**2 - 1)
        p1 = -zeta * wn + sd
        p2 = -zeta * wn - sd
        y = K * (1 - (p2 * np.exp(p1 * t) - p1 * np.exp(p2 * t)) / (p2 - p1))
    else:
        phi = np.arccos(zeta)
        y = K * (1 - np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2) * np.sin(wd * t + phi))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.5))

    # Top: time response
    ax1.plot(t, np.ones_like(t), 'k--', linewidth=1, alpha=0.3, label='Input (step)')
    ax1.plot(t, y, 'b-', linewidth=2, label='x(t)')
    ax1.axhline(K, color='gray', linestyle=':', alpha=0.5,
                label=f'Steady state = {K:.3f} m')
    if zeta < 1 and zeta > 0:
        env = K * (1 + np.exp(-sigma * t) / np.sqrt(1 - zeta**2))
        ax1.plot(t, env, 'r--', linewidth=0.8, alpha=0.4)
        env_low = K * (1 - np.exp(-sigma * t) / np.sqrt(1 - zeta**2))
        ax1.plot(t, env_low, 'r--', linewidth=0.8, alpha=0.4)
    ax1.set_ylabel('Displacement (m)')
    ax1.set_title('Mass-Spring-Damper: Step Response')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8, loc='lower right')

    # Bottom: parameter display
    ax2.axis('off')
    params_text = (
        f'Physical parameters:\n'
        f'  m = {m}, b = {b}, k = {k}\n\n'
        f'Standard-form parameters:\n'
        f'  K = {K:.4f}\n'
        f'  $\\omega_n$ = {wn:.3f} rad/s\n'
        f'  $\\zeta$ = {zeta:.4f}\n'
        f'  $\\omega_d$ = {wd:.3f} rad/s\n'
        f'  $\\sigma$ = {sigma:.3f}')
    ax2.text(0.1, 0.5, params_text, transform=ax2.transAxes,
             fontsize=12, verticalalignment='center',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

def on_run_msd2(b):
    with out_msd2:
        clear_output(wait=True)
        plot_msd_response(m_msd2.value, b_msd2.value, k_msd2.value)

run_msd2.on_click(on_run_msd2)
display(widgets.VBox([m_msd2, b_msd2, k_msd2, run_msd2, out_msd2]))
print("Adjust m, b, k, then click Plot.")

Adjust m, b, k, then click Plot.


---
## Summary

### Transient vs Steady-State
- **Transient response** decays over time, governed by pole locations
- **Steady-state response** persists, following the input shape scaled by $K$

### First-Order Systems
- $H(s) = \dfrac{K}{\tau s + 1}$
- **DC gain $K$**: scales the entire response vertically (transient amplitude *and* steady-state value)
- **Time constant $\tau$**: controls speed only

### Second-Order Systems
- $H(s) = \dfrac{K\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$
- **DC gain $K$**: scales amplitude and steady-state value
- **Natural frequency $\omega_n$**: controls oscillation speed and settling time
- **Damping ratio $\zeta$**: governs overshoot and decay rate
  - Underdamped ($\zeta<1$): oscillatory with exponential decay
  - Critically damped ($\zeta=1$): fastest non-oscillatory
  - Overdamped ($\zeta>1$): slow, non-oscillatory